In [40]:
"""
Heart Disease (Cardiovascular) Prediction Project
Dataset: heart_data.csv (70,000 records)
 
Run:  python heart_disease_project.py
Requires: pip install numpy pandas matplotlib seaborn scikit-learn joblib
"""
 
import warnings
warnings.filterwarnings("ignore")
 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
 
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, f1_score, matthews_corrcoef,
                             classification_report, confusion_matrix,
                             roc_auc_score, roc_curve)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.feature_selection import f_classif, mutual_info_classif, RFE
 
RANDOM_STATE = 42

In [41]:
# 1. LOAD DATA
# ---------------------------------------------------------------
df = pd.read_csv("Dataset/heart_data.csv")
print("Shape:", df.shape)
print(df.head())
print("\nMissing values:\n", df.isnull().sum())

Shape: (70000, 14)
   index  id    age  gender  height  weight  ap_hi  ap_lo  cholesterol  gluc  \
0      0   0  18393       2     168    62.0    110     80            1     1   
1      1   1  20228       1     156    85.0    140     90            3     1   
2      2   2  18857       1     165    64.0    130     70            3     1   
3      3   3  17623       2     169    82.0    150    100            1     1   
4      4   4  17474       1     156    56.0    100     60            1     1   

   smoke  alco  active  cardio  
0      0     0       1       0  
1      0     0       1       1  
2      0     0       0       1  
3      0     0       1       1  
4      0     0       0       0  

Missing values:
 index          0
id             0
age            0
gender         0
height         0
weight         0
ap_hi          0
ap_lo          0
cholesterol    0
gluc           0
smoke          0
alco           0
active         0
cardio         0
dtype: int64


In [42]:
# 2. CLEANING + FEATURE ENGINEERING
# ---------------------------------------------------------------
df = df.drop(columns=[c for c in ["index", "id"] if c in df.columns])
 
# age days mein hai -> saal mein convert
df["age"] = (df["age"] / 365.25).round(1)
 
# Outlier removal
before = len(df)
df = df[(df["ap_hi"].between(70, 250)) &
        (df["ap_lo"].between(40, 150)) &
        (df["ap_hi"] > df["ap_lo"]) &
        (df["height"].between(120, 210)) &
        (df["weight"].between(30, 200))]
print(f"\nOutliers removed: {before - len(df)} rows | Remaining: {len(df)}")
 
#New Feature 
df["bmi"] = df["weight"] / (df["height"] / 100) ** 2
df["pulse_pressure"] = df["ap_hi"] - df["ap_lo"]
 
df.to_csv("datasetCleaned.csv", index=False)
 


Outliers removed: 1390 rows | Remaining: 68610


In [43]:
df.head()

,age,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio,bmi,pulse_pressure
0,50.4,2,168,62.0,110,80,1,1,0,0,1,0,21.967120,30
1,55.4,1,156,85.0,140,90,3,1,0,0,1,1,34.927679,50
2,51.6,1,165,64.0,130,70,3,1,0,0,0,1,23.507805,60
3,48.2,2,169,82.0,150,100,1,1,0,0,1,1,28.710479,50
4,47.8,1,156,56.0,100,60,1,1,0,0,0,0,23.011177,40


In [44]:
df.shape

(68610, 14)

In [45]:
# 3. EDA (graphs save hote hain)
# ---------------------------------------------------------------
plt.figure(figsize=(5, 4))
sns.countplot(x="cardio", data=df)
plt.title("Target distribution (0 = No disease, 1 = Disease)")
plt.tight_layout()
plt.savefig("eda_target.png", dpi=120)
plt.close()
 
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation heatmap")
plt.tight_layout()
plt.savefig("eda_correlation.png", dpi=120)
plt.close()

In [46]:
# 4. TRAIN / TEST SPLIT
# ---------------------------------------------------------------
X = df.drop(columns=["cardio"])
y = df["cardio"]
 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
print("\nTrain:", X_train.shape, "Test:", X_test.shape)


Train: (54888, 13) Test: (13722, 13)


In [47]:
# 4a. FEATURE SELECTION
# ---------------------------------------------------------------
# Goal: keep only the features that really help predict heart disease.
# Important: selection uses ONLY the training data. If the test data were used,
# information from the test set would leak into the model (data leakage) and
# the accuracy would look better than it really is.
#
# Four different methods each pick their top K features. A feature is kept
# only if enough methods agree on it (voting). Using several methods is safer
# than trusting one, because every method has its own blind spots.
print("\n================ FEATURE SELECTION ================")
all_features = list(X_train.columns)
 
TOP_K = 8        # how many features each method may choose
MIN_VOTES = 2   # a feature must be chosen by at least this many methods (out of 4)
 
# Scaled copy of the training data, needed only for RFE with Logistic Regression.
# Features have very different ranges (age ~50, ap_hi ~130, smoke 0/1) and
# Logistic Regression is sensitive to that.
X_tr_scaled = StandardScaler().fit_transform(X_train)
 
# Method 1: ANOVA F-test (statistical, filter method)
# Measures how different the feature's average value is between sick and
# healthy people. Larger score = stronger difference. Detects linear effects only.
f_scores, _ = f_classif(X_train, y_train)
f_rank = pd.Series(f_scores, index=all_features).sort_values(ascending=False)
 
# Method 2: Mutual Information (filter method)
# Measures how much knowing the feature reduces uncertainty about the target.
# Unlike ANOVA it also captures non-linear relationships.
mi_scores = mutual_info_classif(X_train, y_train, random_state=RANDOM_STATE)
mi_rank = pd.Series(mi_scores, index=all_features).sort_values(ascending=False)
 
# Method 3: Random Forest feature importance (tree-based, embedded method)
# Trains a forest and measures how much each feature helped the trees to
# split the data. It also accounts for interactions between features.
rf_sel = RandomForestClassifier(n_estimators=200, max_depth=10, n_jobs=-1,
                                random_state=RANDOM_STATE).fit(X_train, y_train)
rf_rank = pd.Series(rf_sel.feature_importances_,
                    index=all_features).sort_values(ascending=False)
 
# Method 4: RFE - Recursive Feature Elimination (wrapper method)
# Starts with all features, repeatedly removes the weakest one according to a
# Logistic Regression model, until only TOP_K features remain.
rfe = RFE(LogisticRegression(max_iter=1000), n_features_to_select=TOP_K)
rfe.fit(X_tr_scaled, y_train)
rfe_selected = set(np.array(all_features)[rfe.support_])
 
# Voting table: one row per feature, 1 = the method chose it, 0 = it did not.
votes = pd.DataFrame(0, index=all_features,
                     columns=["ANOVA_F", "Mutual_Info", "RandomForest", "RFE"])
votes.loc[f_rank.index[:TOP_K], "ANOVA_F"] = 1
votes.loc[mi_rank.index[:TOP_K], "Mutual_Info"] = 1
votes.loc[rf_rank.index[:TOP_K], "RandomForest"] = 1
votes.loc[list(rfe_selected), "RFE"] = 1
votes["Total_Votes"] = votes.sum(axis=1)          # 0 to 4 votes per feature
votes = votes.sort_values("Total_Votes", ascending=False)
 
print(votes)
votes.to_csv("feature_selection_votes.csv")
 
# Keep the features with enough votes, drop the rest.
selected_features = votes[votes["Total_Votes"] >= MIN_VOTES].index.tolist()
dropped = [f for f in all_features if f not in selected_features]
print(f"\nSelected ({len(selected_features)}): {selected_features}")
print(f"Dropped  ({len(dropped)}): {dropped}")
 
# Bar chart of votes; the red dashed line marks the selection threshold.
plt.figure(figsize=(7, 5))
votes["Total_Votes"].sort_values().plot(kind="barh")
plt.axvline(MIN_VOTES - 0.5, color="red", linestyle="--")
plt.title("Feature Selection - votes from 4 methods")
plt.tight_layout()
plt.savefig("feature_selection.png", dpi=120)
plt.close()
 
# From here on, everything uses only the selected features.
X_train = X_train[selected_features]
X_test = X_test[selected_features]
print("After selection - Train:", X_train.shape, "Test:", X_test.shape)
 


================ FEATURE SELECTION ================
                ANOVA_F  Mutual_Info  RandomForest  RFE  Total_Votes
age                   1            1             1    1            4
ap_hi                 1            1             1    1            4
weight                1            1             1    1            4
cholesterol           1            1             1    1            4
ap_lo                 1            1             1    1            4
bmi                   1            1             1    1            4
pulse_pressure        1            1             1    1            4
gluc                  1            1             0    0            2
height                0            0             1    0            1
active                0            0             0    1            1
gender                0            0             0    0            0
smoke                 0            0             0    0            0
alco                  0            0             0

In [48]:
# 5. MODELS
# ---------------------------------------------------------------
# KNN and Logistic Regression depend on feature scale, so they are wrapped in a
# Pipeline that scales the data first. Tree models do not need scaling.
def scaled(model):
    return Pipeline([("scaler", StandardScaler()), ("model", model)])
 
models = {
    "Logistic Regression": scaled(LogisticRegression(max_iter=1000)),
    "KNN":                 scaled(KNeighborsClassifier(n_neighbors=15)),
    # max_depth / min_samples_leaf limit tree size to prevent overfitting
    # (memorising the training data).
    "Decision Tree":       DecisionTreeClassifier(max_depth=6, random_state=RANDOM_STATE),
    "Random Forest":       RandomForestClassifier(n_estimators=300, max_depth=10,
                                                  min_samples_leaf=5, n_jobs=-1,
                                                  random_state=RANDOM_STATE),
}
 
# Stacking: the predictions of the models above become the input of a final
# Logistic Regression, which learns how to combine them. cv=5 means the base
# predictions come from 5-fold cross-validation, which avoids leakage.
stack = StackingClassifier(
    estimators=[(name, m) for name, m in models.items()],
    final_estimator=LogisticRegression(max_iter=1000),
    cv=5, n_jobs=-1)
models["Stacking"] = stack

In [49]:

# 6. TRAIN + EVALUATE
# ---------------------------------------------------------------
# Metrics:
#   Accuracy : share of correct predictions
#   F1       : balance between precision and recall
#   MCC      : correlation between predictions and truth (-1 to 1), robust metric
#   AUC      : how well the model separates sick from healthy (0.5 = random, 1 = perfect)
# A big gap between Train Acc and Test Acc means overfitting.
results = []
fitted = {}
for name, model in models.items():
    print(f"\nTraining {name} ...")
    model.fit(X_train, y_train)
    fitted[name] = model
 
    tr_pred = model.predict(X_train)
    te_pred = model.predict(X_test)
    te_proba = model.predict_proba(X_test)[:, 1]   # probability of disease
 
    results.append({
        "Model": name,
        "Train Acc": accuracy_score(y_train, tr_pred),
        "Test Acc":  accuracy_score(y_test, te_pred),
        "Test F1":   f1_score(y_test, te_pred),
        "Test MCC":  matthews_corrcoef(y_test, te_pred),
        "Test AUC":  roc_auc_score(y_test, te_proba),
    })
 
results_df = pd.DataFrame(results).sort_values("Test AUC", ascending=False)
print("\n================ MODEL COMPARISON ================")
print(results_df.round(4).to_string(index=False))
results_df.to_csv("model_results.csv", index=False)
 


Training Logistic Regression ...

Training KNN ...

Training Decision Tree ...

Training Random Forest ...

Training Stacking ...

================ MODEL COMPARISON ================
              Model  Train Acc  Test Acc  Test F1  Test MCC  Test AUC
           Stacking     0.7437    0.7340   0.7182    0.4694    0.7988
      Random Forest     0.7472    0.7325   0.7149    0.4667    0.7984
      Decision Tree     0.7340    0.7303   0.7143    0.4619    0.7927
Logistic Regression     0.7253    0.7272   0.7083    0.4563    0.7908
                KNN     0.7491    0.7219   0.7150    0.4438    0.7792


In [50]:
# 7. BEST MODEL: report, confusion matrix, ROC curve
# ---------------------------------------------------------------
best_name = results_df.iloc[0]["Model"]     # model with the highest test AUC
best_model = fitted[best_name]
print(f"\nBest model: {best_name}")
 
y_pred = best_model.predict(X_test)
print(classification_report(y_test, y_pred, target_names=["No Disease", "Disease"]))
 
# Confusion matrix: correct vs wrong predictions for each class.
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["No Disease", "Disease"],
            yticklabels=["No Disease", "Disease"])
plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.title(f"Confusion Matrix - {best_name}")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=120)
plt.close()
 
# ROC curves: the closer a curve is to the top-left corner, the better the model.
plt.figure(figsize=(6, 5))
for name, model in fitted.items():
    fpr, tpr, _ = roc_curve(y_test, model.predict_proba(X_test)[:, 1])
    plt.plot(fpr, tpr, label=name)
plt.plot([0, 1], [0, 1], "k--")      # diagonal = random guessing
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("ROC Curves"); plt.legend()
plt.tight_layout()
plt.savefig("roc_curves.png", dpi=120)
plt.close()
 
# Feature importance from the Random Forest (on the selected features only).
rf = fitted["Random Forest"]
imp = pd.Series(rf.feature_importances_, index=selected_features).sort_values()
plt.figure(figsize=(7, 5))
imp.plot(kind="barh")
plt.title("Feature Importance (Random Forest)")
plt.tight_layout()
plt.savefig("feature_importance.png", dpi=120)
plt.close()


Best model: Stacking
              precision    recall  f1-score   support

  No Disease       0.72      0.78      0.75      6934
     Disease       0.75      0.69      0.72      6788

    accuracy                           0.73     13722
   macro avg       0.74      0.73      0.73     13722
weighted avg       0.74      0.73      0.73     13722



In [52]:
# 8. SAVE MODEL
# ---------------------------------------------------------------
# The list of selected features is saved with the model so that predictions
# later use exactly the same columns in the same order.
joblib.dump({"model": best_model, "features": selected_features},
            "cardio_model.joblib",compress=3)
print("\nModel saved: cardio_model.joblib")


Model saved: cardio_model.joblib


In [53]:
# Predict on a new patient
# ---------------------------------------------------------------
def predict_patient(age_years, gender, height_cm, weight_kg, ap_hi, ap_lo,
                    cholesterol, gluc, smoke, alco, active):
    """
    gender: 1 = female, 2 = male
    cholesterol / gluc: 1 = normal, 2 = above normal, 3 = well above normal
    smoke / alco / active: 0 = No, 1 = Yes
    """
    saved = joblib.load("cardio_model.joblib")
 
    # Build one row with all columns, including the engineered ones,
    # then keep only the features the model was trained on.
    row = pd.DataFrame([{
        "age": age_years, "gender": gender, "height": height_cm,
        "weight": weight_kg, "ap_hi": ap_hi, "ap_lo": ap_lo,
        "cholesterol": cholesterol, "gluc": gluc, "smoke": smoke,
        "alco": alco, "active": active,
        "bmi": weight_kg / (height_cm / 100) ** 2,
        "pulse_pressure": ap_hi - ap_lo,
    }])[saved["features"]]
 
    pred = saved["model"].predict(row)[0]
    prob = saved["model"].predict_proba(row)[0, 1]
    label = "Heart disease risk: YES" if pred == 1 else "Heart disease risk: NO"
    print(f"{label}  (probability = {prob:.1%})")
    return pred, prob
 
 
if __name__ == "__main__":
    print("\n--- Example prediction ---")
    predict_patient(age_years=55, gender=2, height_cm=170, weight_kg=85,
                    ap_hi=150, ap_lo=95, cholesterol=3, gluc=1,
                    smoke=1, alco=0, active=0)


--- Example prediction ---
Heart disease risk: YES  (probability = 84.8%)
